In [0]:
customers = spark.table(
    "databricks_incremental_medallion_lakehouse.silver.customers_silver"
)

orders = spark.table(
    "databricks_incremental_medallion_lakehouse.silver.orders_silver"
)

products = spark.table(
    "databricks_incremental_medallion_lakehouse.silver.products_silver"
)

#join customers and orders

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
customer_sales = (
    customers.alias("c")
    .join(
        orders.alias("o"),
        F.col("c.customer_id") == F.col("o.customer_id"),
        "left"
    )
)

In [0]:
# aggregate

customer_sales_summary = (
    customer_sales
    .groupBy(
        "c.customer_id",
        "c.customer_name",
        "c.city",
        "c.state"
    )
    .agg(
        F.count("o.order_id").alias("total_orders"),
        F.sum("o.amount").alias("total_sales"),
        F.avg("o.amount").alias("average_order_value")
    )
)



In [0]:
customer_sales_summary.display()

In [0]:
customer_sales_summary.printSchema()

In [0]:
%sql
create table databricks_incremental_medallion_lakehouse.gold.customer_sales_summary(
customer_id integer, 
customer_name string,
city string,
state string, 
total_orders long, 
total_sales double, 
average_order_value double)
USING  DELTA;


### OVERWRITE TO GOLD DATABASE

In [0]:
customer_sales_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "databricks_incremental_medallion_lakehouse.gold.customer_sales_summary")